In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
user_df = pd.read_csv("final_df.csv")

In [3]:
user_df.head()
user_df.shape
user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_unique_id        96096 non-null  object 
 1   total_orders              96096 non-null  int64  
 2   total_spent               96096 non-null  float64
 3   avg_price                 95420 non-null  float64
 4   total_items               96096 non-null  int64  
 5   avg_order_value           96096 non-null  float64
 6   days_since_last_purchase  96096 non-null  int64  
 7   customer_tenure_days      96096 non-null  int64  
 8   target_30d                96096 non-null  int64  
 9   target_90d                96096 non-null  int64  
dtypes: float64(3), int64(6), object(1)
memory usage: 7.3+ MB


In [4]:
#creating a new df from user_df
model_df = user_df.copy()

In [5]:
model_df["avg_price"] = model_df["avg_price"].fillna(0)

In [6]:
#removing customer_unique_id so that model won't cheat 
model_df = model_df.drop(
    columns=[
        "customer_unique_id",
        "target_30d",
        "days_since_last_purchase"
    ],
    errors="ignore"
)


In [7]:
#seperate
X = model_df.drop(columns=["target_90d"])
y = model_df["target_90d"]

In [8]:
print(X.info())
print(X.select_dtypes(include="object").columns)
print(y.value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   total_orders          96096 non-null  int64  
 1   total_spent           96096 non-null  float64
 2   avg_price             96096 non-null  float64
 3   total_items           96096 non-null  int64  
 4   avg_order_value       96096 non-null  float64
 5   customer_tenure_days  96096 non-null  int64  
dtypes: float64(3), int64(3)
memory usage: 4.4 MB
None
Index([], dtype='object')
target_90d
0    86432
1     9664
Name: count, dtype: int64


In [9]:
#train test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
#training logistic regression model

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [11]:
#evaluating logistic regression model

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:,1]

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic Regression Results
              precision    recall  f1-score   support

           0       1.00      0.94      0.97     17287
           1       0.65      0.99      0.78      1933

    accuracy                           0.94     19220
   macro avg       0.82      0.97      0.88     19220
weighted avg       0.96      0.94      0.95     19220

ROC-AUC: 0.9903097552350356


In [12]:
#Training Random Forest Classifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [13]:
#Evaluating random forest classifer model
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

print("Random Forest Results")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17287
           1       1.00      0.98      0.99      1933

    accuracy                           1.00     19220
   macro avg       1.00      0.99      0.99     19220
weighted avg       1.00      1.00      1.00     19220

ROC-AUC: 0.9981498257215133


In [14]:
#checking if data leakage has happend

# Shuffle target
y_test_shuffled = np.random.permutation(y_test)

# Evaluate again
roc_auc_score(y_test_shuffled, y_prob_rf)

np.float64(0.5074218697512621)

In [15]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print(importance)

                feature  importance
5  customer_tenure_days    0.956387
0          total_orders    0.013680
4       avg_order_value    0.009512
1           total_spent    0.008357
2             avg_price    0.008216
3           total_items    0.003848


In [16]:
#removing tenure days because it is dominating and the model is only doing this:
#IF tenure_days > threshold → class 0
#ELSE → class 1
X_train_new = X_train.drop(columns=["customer_tenure_days"])
X_test_new = X_test.drop(columns=["customer_tenure_days"])

rf.fit(X_train_new, y_train)
y_pred_new = rf.predict(X_test_new)


In [17]:
y_prob_new = rf.predict_proba(X_test_new)[:, 1]

print("After removing tenure:")
print(classification_report(y_test, y_pred_new))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_new))

After removing tenure:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95     17287
           1       0.52      0.44      0.48      1933

    accuracy                           0.90     19220
   macro avg       0.73      0.70      0.71     19220
weighted avg       0.90      0.90      0.90     19220

ROC-AUC: 0.807441746593248


In [18]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred_new))

[[16488   799]
 [ 1079   854]]


In [19]:
#UPDATED LOGISTIC REGRESSION MODEL WIHTOUT TENURE DAYS

lr = LogisticRegression(class_weight='balanced', max_iter=1000)

lr.fit(X_train_new, y_train)

y_pred_lr_new = lr.predict(X_test_new)
y_prob_lr_new = lr.predict_proba(X_test_new)[:, 1]

print("Logistic Regression WITHOUT tenure:")
print(classification_report(y_test, y_pred_lr_new))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr_new))

Logistic Regression WITHOUT tenure:
              precision    recall  f1-score   support

           0       0.90      0.56      0.69     17287
           1       0.10      0.44      0.16      1933

    accuracy                           0.55     19220
   macro avg       0.50      0.50      0.43     19220
weighted avg       0.82      0.55      0.64     19220

ROC-AUC: 0.5091454271697038


In [20]:
#XGBOOST
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42
)

xgb.fit(X_train_new, y_train)

y_pred_xgb = xgb.predict(X_test_new)
y_prob_xgb = xgb.predict_proba(X_test_new)[:,1]

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

XGBoost Results:
              precision    recall  f1-score   support

           0       0.93      0.62      0.74     17287
           1       0.14      0.57      0.23      1933

    accuracy                           0.62     19220
   macro avg       0.54      0.60      0.49     19220
weighted avg       0.85      0.62      0.69     19220

ROC-AUC: 0.6400341473491664


In [21]:
# PREDICTIONS

# Random Forest (without tenure)
y_pred_rf = y_pred_new  

# Logistic Regression (without tenure)
y_pred_lr = y_pred_lr_new  

# XGBoost (without tenure)
y_pred_xgb = y_pred_xgb  

# models
models = {
    "Random Forest": y_pred_rf,
    "Logistic Regression": y_pred_lr,
    "XGBoost": y_pred_xgb
}

In [22]:
#FINAL METRICS
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for name, y_pred in models.items():
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

df_results = pd.DataFrame(results)
print(df_results)

                 Model  Accuracy  Precision    Recall        F1     TN    FP  \
0        Random Forest  0.902289   0.516636  0.441800  0.476297  16488   799   
1  Logistic Regression  0.549584   0.101186  0.441283  0.164624   9710  7577   
2              XGBoost  0.615297   0.144420  0.573720  0.230753  10717  6570   

     FN    TP  
0  1079   854  
1  1080   853  
2   824  1109  


##### Random Forest performs best overall (Accuracy ≈ 0.90, F1 ≈ 0.48), offering a strong balance between precision and recall.
##### Logistic Regression fails to capture patterns (near-random), while XGBoost improves recall but suffers from many false positives, reducing overall reliability.

In [23]:
#saving best model
import pickle

with open("random_forest_model.pkl", "wb") as f:
    pickle.dump(rf, f)

In [24]:
#saving prediction results
import pandas as pd

results_df = pd.DataFrame({
    "Actual": y_test,
    "RF_Pred": y_pred_rf,
    "LR_Pred": y_pred_lr,
    "XGB_Pred": y_pred_xgb
})

results_df.to_csv("model_predictions.csv", index=False)

##### WE CHOOSE RANDOM FOREST CLASSIFIER AS OUR BEST MODEL

#### BUILDING REGRESSION MODEL FOR predicted_order_value (continuous)

In [25]:
#Defining target variable
model_df["avg_order_value"] = model_df["total_spent"] / model_df["total_orders"]

In [26]:
model_df["avg_order_value"] = (
    model_df["total_spent"] / model_df["total_orders"]
)

model_df["avg_order_value"] = model_df["avg_order_value"].replace([float("inf")], 0)
model_df["avg_order_value"] = model_df["avg_order_value"].fillna(0)

In [27]:
#regression dataset
model_df_reg = model_df[model_df["total_orders"] > 0].copy()

In [28]:
#defining features
features = [
    "total_orders",
    "total_spent",
    "avg_price",
    "total_items",
    "customer_tenure_days"
]

X_reg = model_df_reg[features]
y_reg = model_df_reg["avg_order_value"]

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# X and y
X_reg = model_df_reg[features]
y_reg = model_df_reg["avg_order_value"]

# split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# train regressor
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

rf_reg.fit(X_train_reg, y_train_reg)

# predict
y_pred_reg = rf_reg.predict(X_test_reg)

# evaluate
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("Regression Results")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

Regression Results
MAE: 16.487276178007946
RMSE: 548.3476930794085
R2: 0.640546305092147


The regression model achieved an R² of 0.64, indicating it captures a significant portion of variability in order value. While the average error is low (~16), higher RMSE suggests the presence of outliers or high-value orders affecting predictions.

In [30]:
model_df_reg["predicted_order_value"] = rf_reg.predict(X_reg)
model_df_reg[["avg_order_value", "predicted_order_value"]].head()

,avg_order_value,predicted_order_value
0,141.90,128.269074
1,27.19,47.739509
2,86.22,83.319151
3,43.62,47.739509
4,196.89,190.416021


The regression model produces predictions close to actual order values, particularly for medium and high-value customers. Some variance exists for smaller orders, but overall performance is strong and consistent with the R² score of ~0.64.

In [36]:
final_dataframe = model_df.copy()

# Propensity (for ALL users)
final_dataframe["propensity_score"] = rf.predict_proba(
    model_df[X_train_new.columns]
)[:, 1]

# Order value (only for users with orders)
final_dataframe["predicted_order_value"] = np.nan
final_dataframe.loc[model_df_reg.index, "predicted_order_value"] = model_df_reg["predicted_order_value"]

# Business score
final_dataframe["customer_score"] = (
    final_dataframe["propensity_score"] * final_dataframe["predicted_order_value"]
)

final_dataframe.head()

,total_orders,total_spent,avg_price,total_items,avg_order_value,customer_tenure_days,target_90d,propensity_score,predicted_order_value,customer_score
0,1,141.90,129.90,1,141.90,160,0,0.005000,128.269074,0.641345
1,1,27.19,18.90,1,27.19,163,0,0.000000,47.739509,0.000000
2,1,86.22,69.00,1,86.22,585,0,0.013583,83.319151,1.131759
3,1,43.62,25.99,1,43.62,369,0,0.000000,47.739509,0.000000
4,1,196.89,180.00,1,196.89,336,0,0.005000,190.416021,0.952080


In [37]:
#showing top customers who are most likley to buy
final_dataframe.sort_values(by="customer_score", ascending=False).head(10)

,total_orders,total_spent,avg_price,total_items,avg_order_value,customer_tenure_days,target_90d,propensity_score,predicted_order_value,customer_score
75269,4,27935.46,170.000000,24,6983.865,71,1,0.685,28117.727717,19260.643486
75528,1,11572.80,113.830000,10,11572.800,66,1,0.655,10482.049211,6865.742233
85697,1,9751.00,359.000000,5,9751.000,86,1,0.695,8513.684982,5917.011062
45831,1,8511.75,330.000000,5,8511.750,71,1,0.665,8031.027509,5340.633293
54401,1,8780.50,330.000000,5,8780.500,81,1,0.650,8054.857596,5235.657437
54182,2,11881.01,262.142857,7,5940.505,104,1,0.635,7269.230307,4615.961245
26205,1,6922.21,6729.000000,1,6922.210,83,1,0.640,6525.389708,4176.249413
84230,1,6265.15,220.526000,5,6265.150,72,1,0.655,6074.857190,3979.031459
51636,1,6035.04,150.000000,6,6035.040,154,0,0.650,6055.065645,3935.792669
36752,1,5988.60,150.000000,6,5988.600,78,1,0.665,5887.209472,3914.994299


In [38]:
#saving top 10 customers in csv
top10 = final_dataframe.sort_values(
    by="customer_score", ascending=False
).head(10)

top10.to_csv("top_10_customers.csv", index=False)